# Explore the platform

A local Parquet lake of five years of daily US equity data from Massive: OHLCV
bars, the per-session universe, corporate actions and the FINRA short datasets.
DuckDB queries it, dbt builds the derived tables. The design problem is
point-in-time correctness: every read answers "what was knowable on date D?".

Every read goes through `sdp.dal`, which returns a lazy `DuckDBPyRelation`.
Materialise at the edge with `.pl()` or `.fetchall()`. Each section is independent.

In [ ]:
import datetime as dt

import polars as pl

from sdp import dal

pl.Config.set_tbl_rows(15)
pl.Config.set_fmt_str_lengths(60)

con = dal.con()   # The one connection that owns every relation.
print(dal.status())

---
## 1. What is published

A partition exists only after its audit passed, so a missing date means "no
data", not "bad data". Event streams carry a partition count and a date span;
the corporate action tables carry a row count and no dates.

In [ ]:
# dal.status(): one line per dataset, either partition count and span, or a
# row count for a current-state table.
print(dal.status())

In [ ]:
# gaps() lists the XNYS sessions inside a range that have no partition.
# An empty list means the range is complete.
first, last = dal.coverage(dal.DAY_AGGS)
print("day aggregates", first, "to", last)
print("interior gaps:", dal.gaps(dal.DAY_AGGS, first, last) or "none")

---
## 2. Point-in-time, and where it applies

The event streams carry the guarantee in their storage: a partition is one
immutable session. The ticker reference is the important case, and it is what
frees the universe of survivorship bias (section 5).

The corporate action tables do not, on purpose. The endpoint has no `as_of`, so
it gives the vendor's belief now. Read it with `dal.current()`, or
`dal.splits()` / `dal.dividends()`.

In [ ]:
# One table, no date in the path, no as_of.
splits = dal.splits()
print("split rows:", con.sql("select count(*) from splits").fetchone()[0])
print("built from the vendor pull of:",
      con.sql("select distinct vendor_pull_date from splits").fetchone()[0])

# vendor_pull_date is provenance, not a key: it names the vendor file that
# built the table, so a study can rebuild it.
dal.splits().limit(3).pl()

In [ ]:
# The two kinds cannot be confused. Each function refuses the wrong kind, and
# says which one to use instead.
for call, label in [
    (lambda: dal.series(dal.SPLITS), "series() on a current-state dataset"),
    (lambda: dal.current(dal.DAY_AGGS), "current() on an event stream"),
]:
    try:
        call()
    except ValueError as exc:
        print(f"{label}:\n  {exc}\n")

### What the current table cannot tell you

A restated factor overwrites the old value and leaves no trace in `raw/`. It is
recoverable from `vendor/`, which keeps every pull dated:
`ca.rebuild(dataset, pull_date)` builds the table as any kept pull stated it.

In [ ]:
from sdp.ingest import massive_corporate_actions as ca

# Every vendor pull that is kept. This archive is the reason dropping the
# pull_date partitions costs capability rather than information.
for pull, path in ca.vendor_pulls("massive_splits"):
    print(f"{pull}  {path.stat().st_size / 1e6:8.1f} MB  {path.name}")

In [ ]:
# The published table against what an older pull said. rebuild() would replace
# raw/, so read the vendor file directly here to leave the lake alone.
old_pull, old_path = ca.vendor_pulls("massive_splits")[0]
con.sql(f'''
    with older as (
        select * from read_json('{old_path}',
                                format='newline_delimited', sample_size=-1)
    )
    select
        (select count(*) from splits) as rows_now,
        (select count(*) from older)  as rows_in_the_pull_of_{old_pull:%Y_%m_%d}
''').pl()

---
## 3. Restatement, measured from the vendor archive

A restatement is the vendor changing what it says about the past. It matters
because the factor is cumulative, so a revision changes every adjusted price
before that event. The only way to see one is to compare two full pulls, which
`sdp.restatement` does over `vendor/`.

In [ ]:
# Register the oldest and newest kept pulls as views over vendor/.
pulls = dict(ca.vendor_pulls("massive_splits"))
old_pull, new_pull = min(pulls), max(pulls)
print(f"comparing the pull of {old_pull} against {new_pull}")

for name, pull in [("older", old_pull), ("newer", new_pull)]:
    con.execute(f"""
        create or replace temp view {name} as select * from
        read_json('{pulls[pull]}', format='newline_delimited', sample_size=-1)
    """)

con.sql("""
    select (select count(*) from older) as rows_older,
           (select count(*) from newer) as rows_newer
""").pl()

The row count moved. Now find what changed. The obvious key is the vendor
`id`, so try that first.

In [ ]:
con.sql('''
    select
        (select count(*) from older a
          where not exists (select 1 from newer b where b.id = a.id)
        ) as ids_only_in_the_old_pull,
        (select count(*) from newer b
          where not exists (select 1 from older a where a.id = b.id)
        ) as ids_only_in_the_new_pull
''').pl()

That reads as though hundreds of corporate actions disappeared. They did not.

Check whether the same event is still there under a different `id`, by matching
on `(ticker, execution_date)` instead.

In [ ]:
con.sql('''
    with dropped as (
        select a.* from older a
        where not exists (select 1 from newer b where b.id = a.id)
    )
    select
        count(*) as ids_that_vanished,
        count(*) filter (
            exists (select 1 from newer n
                    where n.ticker = d.ticker and n.execution_date = d.execution_date)
        ) as same_event_still_present_under_a_new_id
    from dropped d
''').pl()

**Almost every one is still present under a different id.** The vendor `id` is
not stable across pulls, so a diff on `id` overstates the churn and nothing
durable may key on it. Use the event key. Now measure the change that is real.

In [ ]:
# Restrict to keys unambiguous in both pulls, or the join fans out across the
# duplicated split keys and overstates the count.
con.sql('''
    with ua as (select ticker, execution_date from older
                group by 1, 2 having count(*) = 1),
         ub as (select ticker, execution_date from newer
                group by 1, 2 having count(*) = 1),
         k  as (select * from ua intersect select * from ub)
    select
        (select count(*)
           from older a join newer b using (ticker, execution_date)
                        join k using (ticker, execution_date)
          where a.historical_adjustment_factor
                is distinct from b.historical_adjustment_factor) as factor_restated,
        (select count(*) from newer b
          where not exists (select 1 from older a
                            where a.ticker = b.ticker
                              and a.execution_date = b.execution_date)) as genuinely_new_events
''').pl()

A restated factor is not cosmetic: it changes every adjusted price before that
event. This is what current-state storage gives up, and the discipline that
replaces the old policy is one line: a study records the `vendor_pull_date` of
the table it used.

In [ ]:
# The tickers that were restated. These are the names to look at first.
con.sql('''
    with ua as (select ticker, execution_date from older
                group by 1, 2 having count(*) = 1),
         ub as (select ticker, execution_date from newer
                group by 1, 2 having count(*) = 1),
         k  as (select * from ua intersect select * from ub)
    select a.ticker, a.execution_date, a.adjustment_type,
           a.historical_adjustment_factor as factor_before,
           b.historical_adjustment_factor as factor_after
    from older a join newer b using (ticker, execution_date)
                 join k using (ticker, execution_date)
    where a.historical_adjustment_factor
          is distinct from b.historical_adjustment_factor
    order by a.execution_date desc
    limit 15
''').pl()

---
## 4. Day aggregates

Unadjusted prices, one row for each ticker and session. Unadjusted is the point.
An adjusted price restates retroactively, and that would break the immutability of
a published partition.

In [ ]:
bars = dal.day_aggs()
con.sql("select count(*) as rows, count(distinct ticker) as tickers, "
        "min(date) as first_session, max(date) as last_session from bars").pl()

In [ ]:
con.sql('''
    select date, count(*) as tickers,
           round(sum(volume * close) / 1e9, 1) as dollar_volume_bn
    from bars group by date order by date
''').pl()

A detail worth knowing before you trust a type: `volume` is a floating point
number and not an integer. Most rows are fractional, because the consolidated tape
carries fractional share quantities.

In [ ]:
con.sql('''
    select count(*) as rows,
           count(*) filter (volume <> floor(volume)) as fractional_volume,
           round(100.0 * count(*) filter (volume <> floor(volume)) / count(*), 1) as pct
    from bars
''').pl()

---
## 5. The universe, and the identifier that is still an open question

Two independent filters build the universe, and both are recomputed for each date.
The instrument filter is the simple one.

In [ ]:
names = dal.tickers_on(dal.partitions(dal.TICKERS)[-1])
con.sql('''
    select type, count(*) as n
    from names group by type order by n desc limit 12
''').pl()

In [ ]:
con.sql('''
    select
        count(*) as all_instruments,
        count(*) filter (type = 'CS') as common_stock,
        count(*) filter (type = 'CS'
                         and primary_exchange in ('XNYS','XNAS','XASE')) as after_exchange_filter
    from names
''').pl()

### The identifier problem

A ticker symbol changes, and worse, it is reused: a freed symbol can go to a
different company, splicing two return series into one fat-tailed series. The
intended key was `composite_figi`, but its coverage blocks it (section 9).

In [ ]:
con.sql('''
    select
        count(*) as cs_rows,
        count(*) filter (composite_figi is null) as null_composite_figi,
        count(*) filter (share_class_figi is null) as null_share_class_figi,
        count(*) filter (cik is null) as null_cik
    from names where type = 'CS'
''').pl()

In [ ]:
# The landscape is the inverse of what you would expect. FIGI is complete for
# ETFs and patchy for common stock. CIK is the other way round.
con.sql('''
    select type, count(*) as n,
           count(*) filter (composite_figi is null) as null_figi,
           count(*) filter (cik is null) as null_cik
    from names
    where type in ('CS', 'ETF', 'ADRC', 'WARRANT', 'PFD')
    group by type order by n desc
''').pl()

The number that decides this question is the exposure **after** the liquidity
filter, not before it. If the null-FIGI names are a tail of recent listings and
shells, the liquidity filter removes them and the question does not matter. That
diagnostic needs the backfill.

---
## 6. Adjustment for corporate actions

The vendor's `historical_adjustment_factor` is cumulative, so adjustment is one
as-of lookup, not a chained product: for a price on D, take the first event
after D and multiply. Check it against AAPL, which split 2:1 in 2005, 7:1 in
2014 and 4:1 in 2020.

In [ ]:
splits = dal.splits()
con.sql('''
    select execution_date, adjustment_type,
           split_from, split_to,
           split_to / split_from as ratio,
           historical_adjustment_factor
    from splits where ticker = 'AAPL' order by execution_date
''').pl()

In [ ]:
# The 2005 factor must equal 1/2 * 1/7 * 1/4 = 1/56, compounded with the later
# two splits. Reproducing it from the ratios confirms the semantics.
expected = 1 / (2 * 7 * 4)
print(f"1 / (2 * 7 * 4) = {expected:.6f}")

actual = con.sql(
    "select historical_adjustment_factor from splits "
    "where ticker = 'AAPL' and execution_date = date '2005-02-28'"
).fetchone()[0]
print(f"vendor factor for 2005-02-28 = {actual}")
print("match:", round(expected, 6) == round(actual, 6))

### The boundary is strict

Split adjustment applies overnight. On the execution date all trading is already
adjusted, including the pre-market session. The join must be `> D` and not `>= D`.
An error of one day makes one very large false return for each split and each
name.

In [ ]:
# The as-of join, written out. This is the shape that the dbt staging model needs.
con.sql('''
    with universe as (
        select ticker, date, close from bars where ticker = 'AAPL'
    )
    select u.date, u.close,
           (select s.historical_adjustment_factor
              from splits s
             where s.ticker = u.ticker
               and s.execution_date > u.date      -- strictly after
             order by s.execution_date
             limit 1) as split_factor
    from universe u
    order by u.date
    limit 10
''').pl()

A null factor means that no split follows that date, so the price needs no split
adjustment. AAPL has no split after 2020, so recent bars are already on today's
share basis.

---
## 7. The audit invariants, checked against the live data

These are the properties that the audits enforce at ingest. Confirm that they hold
on what is published.

In [ ]:
# Splits classification. Every forward_split has a ratio above 1, every
# reverse_split below 1, and every stock_dividend above 1.
con.sql('''
    select adjustment_type,
           count(*) as n,
           min(split_to / split_from) as min_ratio,
           max(split_to / split_from) as max_ratio
    from splits group by adjustment_type order by n desc
''').pl()

In [ ]:
# Reverse splits outnumber forward splits by about two to one. Most are
# distressed microcaps doing a 1-for-10 to keep a listing.
con.sql('''
    select adjustment_type, count(*) as n,
           round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
    from splits group by adjustment_type order by n desc
''').pl()

In [ ]:
# RYCEF: a Rolls-Royce ADR with a 1:72 stock dividend and factor 0.0. The C
# shares are not fungible, so no valid adjustment exists. Treating it as a
# split would fake a one-day -98.6%.
con.sql('''
    select ticker, execution_date, adjustment_type, split_from, split_to,
           historical_adjustment_factor
    from splits
    where historical_adjustment_factor <= 0
    order by execution_date desc
    limit 10
''').pl()

In [ ]:
# The dividend factor is different. A null there is structural and not a defect.
# It means the vendor had no price on the ex-date to compute (1 - D/P) against.
divs = dal.dividends()
con.sql('''
    select count(*) as rows,
           count(*) filter (historical_adjustment_factor is null) as null_factor,
           round(100.0 * count(*) filter (historical_adjustment_factor is null)
                 / count(*), 1) as pct_null,
           count(*) filter (currency is not null and currency <> 'USD') as not_usd
    from divs
''').pl()

In [ ]:
# Restrict to tickers that trade on the ingested tape. The nulls are dividends
# on securities that never trade there (foreign issuers, OTC, fund classes).
con.sql('''
    with traded as (select distinct ticker from bars)
    select
        case when d.ticker in (select ticker from traded)
             then 'in the day aggregates' else 'not in the day aggregates' end as group_,
        count(*) as rows,
        round(100.0 * count(*) filter (d.historical_adjustment_factor is null)
              / count(*), 1) as pct_null_factor
    from divs d group by 1
''').pl()

Before the backfill the bars covered a few weeks, so any earlier-delisted name
read as "not traded", the survivorship-sensitive set. With five years of bars
the null rate is clean: **0.97%** inside the traded set against **39.08%**
outside. So `adj_close_total` is the default column, not the fallback.

---
## 8. The universe, now that the backfill has run

The lake holds 1,255 sessions from 2021-08-23. Everything below is the first
look at the models on real history rather than on three weeks.

In [ ]:
import duckdb

from sdp.config import settings

# The warehouse is dbt's output. Read it read-only, so a build can run beside
# this notebook.
wh = duckdb.connect(str(settings.warehouse_path), read_only=True)

wh.sql('''
    select date, count(*) filter (in_universe) as names
    from main_staging.stg_universe
    group by 1 order by 1
''').pl()

In [ ]:
# The size of the universe over time. The target was 1,000 to 2,000 names.
wh.sql('''
    with per_date as (
        select date, count(*) filter (in_universe) as n
        from main_staging.stg_universe group by 1
    )
    select date_trunc('year', date) as year,
           round(avg(n)) as avg_names,
           min(n) as min_names,
           max(n) as max_names
    from per_date group by 1 order by 1
''').pl()

Two things to read.

**The universe is ~2,960 names at the median, half again above the 1,000-2,000
target.** The thresholds are dbt vars, so tightening `min_dollar_volume` is a
dial, not a defect.

**The first 59 sessions have no universe:** a name needs 60 sessions of history,
which the start of the lake lacks. The usable window begins 2021-11-15 (1,196
sessions). Say so in a writeup.

In [ ]:
# Where the names are lost. Each filter is a column, so an empty universe is
# diagnosable instead of mysterious.
wh.sql('''
    select
        count(*)                                  as rows,
        count(*) filter (passes_instrument)       as after_instrument,
        count(*) filter (passes_instrument and passes_price)      as and_price,
        count(*) filter (passes_instrument and passes_price
                         and passes_adv)          as and_adv,
        count(*) filter (in_universe)             as in_universe
    from main_staging.stg_universe
    where date = (select max(date) from main_staging.stg_universe)
''').pl()

---
## 9. The open questions, answered

`python -m sdp.diagnostics` runs these against the whole lake. The queries are
repeated here so the numbers sit next to the reasoning.

### The identifier

Ticker reuse is real, and its size decides whether a simple key is defensible.

In [ ]:
names = dal.tickers()

con.sql('''
    with per_ticker as (
        select ticker, count(distinct composite_figi) as figis
        from names where type = 'CS' and composite_figi is not null
        group by 1
    )
    select count(*) as cs_tickers,
           count(*) filter (figis > 1) as more_than_one_figi,
           count(*) filter (figis > 2) as more_than_two
    from per_ticker
''').pl()

886 of 9,018 common stock tickers carry more than one FIGI across the window,
one in ten, so keying on the symbol would splice two companies into one series.
The number that decides the fallback rule is the FIGI gap after the liquidity
screen, not before.

In [ ]:
wh.sql('''
    select
        count(*) as cs_rows,
        round(100.0 * count(*) filter (security_key is null
                                       or key_rule <> 'share_class_figi')
              / count(*), 2) as pct_not_on_figi,
        count(*) filter (in_universe) as after_the_screen,
        round(100.0 * count(*) filter (in_universe
                                       and key_rule <> 'share_class_figi')
              / nullif(count(*) filter (in_universe), 0), 2) as pct_not_on_figi_in_universe
    from main_staging.stg_universe
    where type = 'CS'
''').pl()

The screen helps but does not rescue it: the gap falls from 15.4% to **9.9%**,
still one row in ten falling back to CIK or ticker. So the coalesce with a
recorded `key_rule` is the right answer, and a study should report results with
and without the fallback rows.

### The ambiguous event key

In [ ]:
splits = dal.splits()

con.sql('''
    with per_key as (
        select ticker, execution_date,
               count(*) as rows,
               count(distinct historical_adjustment_factor) as factors
        from splits group by 1, 2
    )
    select count(*) as distinct_keys,
           count(*) filter (rows > 1) as duplicated,
           count(*) filter (rows > 1 and factors > 1) as and_disagreeing
    from per_key
''').pl()

209 split keys are duplicated and every one disagrees about the factor; 33 fall
inside the price window. Dividends are worse: 13,864 duplicated keys, 7,449
disagreeing, 3,773 in the window. An as-of join against the raw rows fans out,
so `stg_corporate_actions` resolves the key first. The next cell shows the
failure on purpose.

In [ ]:
# The trap, demonstrated. Joining prices to the raw splits on the event key
# multiplies rows wherever the key is duplicated.
con.sql('''
    with one_name as (
        select ticker, execution_date
        from splits group by 1, 2 having count(*) > 1 limit 1
    )
    select s.ticker, s.execution_date, s.split_from, s.split_to,
           s.historical_adjustment_factor
    from splits s join one_name using (ticker, execution_date)
''').pl()

---
## 10. Restatement, measured over two vendor pulls

`python -m sdp.restatement` compares the oldest kept pull against the newest,
and separates the vendor correcting itself from the cumulative factor behaving
as designed.

In [ ]:
from sdp import restatement

# sdp.restatement reads vendor/, so it keeps working however raw/ is stored.
print(restatement.diff(dal.SPLITS))

Read on the vendor `id`, hundreds of events appear and disappear in two weeks.
Almost none of that is real. The `id` is not stable across pulls, so the same
event returns under a new one. Match on the event instead.

In [ ]:
# The event diff, written out, to show what the module does internally.
con.sql('''
    with ka as (select distinct ticker, execution_date from older),
         kb as (select distinct ticker, execution_date from newer)
    select
        (select count(*) from ka where not exists
            (select 1 from kb where kb.ticker = ka.ticker
                                and kb.execution_date = ka.execution_date)) as events_gone,
        (select count(*) from kb where not exists
            (select 1 from ka where ka.ticker = kb.ticker
                                and ka.execution_date = kb.execution_date)) as events_new
''').pl()

5 gone and 82 new on the event key, against 400 and 479 on the id, so the rest
was churn.

### How much of the restatement is mechanical

The factor is cumulative, so a new dividend changes every earlier factor for
that ticker by design. That is not the vendor correcting itself, and the two
must be separated. The join below keeps only keys unambiguous in both pulls, or
it fans out across the duplicated dividend keys and overstates the count.

In [ ]:
d_pulls = dict(ca.vendor_pulls("massive_dividends"))
d_old, d_new = min(d_pulls), max(d_pulls)
for name, pull in [("d_older", d_old), ("d_newer", d_new)]:
    con.execute(f"""
        create or replace temp view {name} as select * from
        read_json('{d_pulls[pull]}', format='newline_delimited', sample_size=-1)
    """)

con.sql(f'''
    with ua as (select ticker, ex_dividend_date from d_older
                group by 1, 2 having count(*) = 1),
         ub as (select ticker, ex_dividend_date from d_newer
                group by 1, 2 having count(*) = 1),
         k  as (select * from ua intersect select * from ub),
         changed as (
             select a.ticker
             from d_older a join d_newer b using (ticker, ex_dividend_date)
                            join k using (ticker, ex_dividend_date)
             where a.historical_adjustment_factor
                   is distinct from b.historical_adjustment_factor
         ),
         went_ex as (
             select distinct ticker from d_newer
             where ex_dividend_date > date '{d_old}'
               and ex_dividend_date <= date '{d_new}'
         )
    select count(*) as restated,
           count(*) filter (ticker in (select ticker from went_ex)) as mechanical,
           count(*) filter (ticker not in (select ticker from went_ex)) as vendor_revision
    from changed
''').pl()

**98.1% of it is mechanical.** 87,267 of 88,972 restated factors sit on a
ticker that went ex inside the window, the cumulative factor doing its job. Only
1,705 rows are the vendor changing its mind. That is the number for a writeup:
the lookahead risk is dominated by an understood mechanism, not arbitrary
revision.

---
## 11. A typical workflow

The loop from here on. Nothing below builds a signal worth trading. The point is
the shape of the loop, and finding the plumbing errors on a result nobody is
attached to.

**1. Pull the universe for a date range, from the model and never by hand.**

In [ ]:
panel = wh.sql('''
    select u.date, u.security_key, u.ticker, u.close, u.adv,
           p.adj_close_total
    from main_staging.stg_universe u
    join main_staging.stg_prices_adjusted p
      on p.date = u.date and p.ticker = u.ticker
    where u.in_universe
      and u.date >= date '2024-01-01'
''')
print(f"{panel.count('*').fetchone()[0]:,} name-days")
panel.limit(5).pl()

**2. Build a signal.** A deliberately weak five-day reversal, with no
residualization. The point is to exercise the harness on a result nobody is
attached to and find plumbing errors.

In [ ]:
signal = wh.sql('''
    with px as (
        select u.date, u.security_key, p.adj_close_total as px
        from main_staging.stg_universe u
        join main_staging.stg_prices_adjusted p
          on p.date = u.date and p.ticker = u.ticker
        where u.in_universe and p.adj_close_total is not null
    ),
    with_lags as (
        select date, security_key, px,
               lag(px, 5) over w as px_5,
               lead(px, 1) over w as px_next
        from px
        window w as (partition by security_key order by date)
    )
    select date, security_key,
           -- The signal. Negative of the trailing five-day return.
           -(px / px_5 - 1)          as reversal,
           -- The thing it must predict. The next day return.
           px_next / px - 1          as fwd_return
    from with_lags
    where px_5 is not null and px_next is not null
''')
print(f"{signal.count('*').fetchone()[0]:,} rows with a signal and a forward return")

Two things in that query are the whole point.

`lead(px, 1)` is the forward return, the only place the future may appear. If a
signal column ever reads `lead`, the result is meaningless and will look
excellent. The partition is `security_key`, not `ticker`, so a renamed symbol
does not lag two companies into one series.

**3. Score it** with the information coefficient: the cross-sectional rank
correlation between signal and forward return, per date, then averaged.

In [ ]:
# `signal` was built on `wh`, and a relation belongs to the connection that
# made it. Querying it from `con` raises.
ic = wh.sql('''
    with daily as (
        select date, corr(rs, rf) as ic
        from (
            select date,
                   rank() over (partition by date order by reversal)   as rs,
                   rank() over (partition by date order by fwd_return) as rf
            from signal
        )
        group by date
        having count(*) > 100
    )
    select count(*)                      as days,
           round(avg(ic), 5)             as mean_ic,
           round(stddev(ic), 5)          as sd_ic,
           round(avg(ic) / (stddev(ic) / sqrt(count(*))), 2) as t_stat
    from daily
''')
ic.pl()

Treat that number as a plumbing check, not a result. What is still missing:

- **Residualization** — a raw reversal is mostly a bet against the market and
  sectors; the PCA risk model comes first.
- **Purging** — overlapping windows leak across a naive train/test split.
- **Costs** — reversal turns over daily and is the strategy most easily
  destroyed by the spread.
- **Bid-ask bounce** — consecutive closes in a wide-spread name alternate
  between bid and ask, manufacturing negative serial correlation that cannot be
  harvested. The honest check is whether the IC survives a much tighter ADV floor.

**4. Vary the thresholds** as the robustness check:

```bash
python -m sdp.transform build --vars '{min_dollar_volume: 10000000}'
```

If the IC is 0.04 at a \$1M floor and 0.005 at \$10M, the signal is an
unharvestable illiquidity premium. That comparison is the result, not the first
number.

**5. Write the kill-log entry whatever happens.**

---
## 12. Where things stand

Built and measured: 1,255 sessions of bars and tickers from 2021-08-23, 638 of
short volume, 119 short-interest settlements, no gaps; the staging models; a
usable universe of ~2,960 names a day from 2021-11-15.

Open:

- **The universe is half again above target.** Tighten `min_dollar_volume` first.
- **The identifier fallback is 9.9% after the screen.** The coalesce rule can be
  settled, reporting with and without the fallback rows.
- **The strategic choice.** US cross-sectional daily stat-arb is crowded. Short
  interest as a crowding signal has the full price window and is the study almost
  nobody else can write.

Next: the evaluation harness, then the PCA risk model, then residual reversal.